<div style="text-align: center">

# Europe Crop Yield Regression — FAO DATA
</div>

**Dataset:** FAOSTAT — Food and Agriculture Organization of the United Nations  
**Source:** https://www.fao.org/faostat/


## Table of Contents
1. [Introduction](#1.-Introduction)
2. [Setup & Data Loading](#2.-Setup-&-Data-Loading)
3. [Exploratory Data Analysis](#3.-Exploratory-Data-Analysis)
4. [Modelling](#4.-Modelling)
5. [Model Diagnostics](#5.-Model-Diagnostics)
6. [Interpretation](#6.-Interpretation)
7. [Limitations](#7.-Limitations)
8. [Conclusion](#8.-Conclusion)
9. [Citations](#9.-Citations)

---
## 1. Introduction
Agriculture sector plays a crucial role in achieving the UN Sustainable Development Goals, particularly food security.
In 2021, the Food and Agriculture Organization estimated that 2.3 billion people were food insecure (FAO et al., 2023). 
Insufficient nutrition poses a significant challenge to immediate survival, physical activity, and overall health. Moreover, agriculture constitutes the main economic activity in Low Income Countries (LICs), contributing over 40% of GDP (World Bank, 2022).
In particular, crop yield, the quantity of food produced per unit of land, is a key determinant of food security, rural incomes, land-use pressure, and agricultural greenhouse gas emissions. Yield varies substantially across countries and over time, driven by inputs such as fertilizers and pesticides, technology adoption, climate, and institutional factors (Tilman et al., 2002).
This project focuses on wheat yield in the European Union, using three datasets from FAOSTAT (FAO, 2023):
- Crops & Livestock products | Yield (kg/ha), area harvested, production | 1961–2024, 173 crops, 198 countries 
- Fertilizers by Nutrient | Nitrogen, phosphate, potash use (kg/ha) | 1961–2023, 198 countries
- Pesticides Use | Total pesticide use (tonnes) | 1990–2023, 198 countries

Wheat was selected because it is the most produced cereal in EU27 and one of only two cereals cultivated in all 27 member states, making it well-suited for a cross-country regression analysis.

**Research question**: What drives differences in wheat yield across EU27 countries, and how much of the variance is explained by nitrogen use, harvested area, and pesticide use?

---
## Setup & Data Loading

In [ ]:
import sys, os

def _find_project_root():
    notebook_dir = os.path.dirname(os.path.abspath('fao_crop_yield_analysis.ipynb'))
    if os.path.isdir(os.path.join(notebook_dir, 'src')):
        return notebook_dir
    parent = os.path.dirname(notebook_dir)
    if os.path.isdir(os.path.join(parent, 'src')):
        return parent
    raise FileNotFoundError(f'Cannot find src/ in {notebook_dir} or {parent}')

PROJECT_ROOT = _find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

from src.data_loader import (
    get_fao_data, load_eu_raw, get_feature_target_split,
    TARGET, FOCUS_CROPS, EU27_COUNTRIES
)
from src.visualization import (
    plot_top_eu_crops,
    plot_wheat_map_eu27,
    plot_yield_vs_nitrogen,
    plot_correlation_heatmap,
    plot_actual_vs_predicted,
)

from src.modelling import get_ols_features, fit_ols, evaluate_model

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

fao, oh_encoded_variable_names = get_fao_data(
    data_dir=DATA_DIR,
    year_min=1990,
    year_max=2023,
    categorical_variable_list=['Item'],
)

print(f'Dataset: {fao.shape[0]:,} rows x {fao.shape[1]} columns')
print(f'Countries: {fao["Area"].nunique()} | Crop: Wheat only')
fao.head()

---
## 3. Exploratory Data Analysis

### 3.1 Crop Selection

In [ ]:
# Load the full EU27 dataset (all crops) to justify selecting Wheat
qcl_eu_raw = load_eu_raw(data_dir=DATA_DIR)

print(f'Total crops in EU27: {qcl_eu_raw["Item"].nunique()}')
fig = plot_top_eu_crops(qcl_eu_raw, focus_crops=FOCUS_CROPS, top_n=10)
plt.show()

The three panels justify the choice of wheat as the focus crop for the log-log OLS regression. The left panel shows that wheat is the most produced cereal in EU27 by total tonnage. Wheat and barley are the only cereals cultivated in all 27 EU member states. Among these two, wheat produces roughly twice the total tonnage, making it the dominant cereal and the natural choice for a cross-country regression analysis. 

### 3.2 Wheat Yield Across EU27 Countries

In [ ]:
fig = plot_wheat_map_eu27(fao, variable='Yield_t_ha')
plt.show()

The map reveals a clear geographic pattern in EU wheat yield. Western and Northern European countries (Netherlands, Germany, France, UK) consistently achieve yields above 6-7 t/ha, while Eastern European countries (Romania, Bulgaria, Latvia) tend to have lower yields around 3-4 t/ha. This North-West vs East gradient reflects differences in input intensity, technology adoption, and farming infrastructure.

### 3.3 Yield vs Nitrogen Use

In [ ]:
fig = plot_yield_vs_nitrogen(fao, top_crops=1)
plt.show()

The scatter plot shows a positive relationship between nitrogen use and wheat yield across EU27 countries. The relationship shows some diminishing returns at very high nitrogen levels, where EU countries already apply intensive fertilization. This motivates the log-transformation of nitrogen use in the regression.

### 3.4 Correlation Heatmap

In [ ]:
fig = plot_correlation_heatmap(fao)
plt.show()

The heatmap confirms that nitrogen use is the strongest predictor of yield in the EU27 sample. Pesticide use also shows a positive correlation. Area harvested has a weak negative correlation — larger harvested areas in countries like France, Poland, and Romania include more marginal land, pulling down the average yield per hectare.

---
## 4. Modelling — OLS Regression

This analysis uses **Ordinary Least Squares (OLS)** regression on log-transformed wheat yield. 

Since raw yield is right-skewed, I log-transform the target. This also makes coefficients interpretable as **elasticities**: a 1% increase in an input is associated with a β% change in yield.

The model estimated is:

$$\log(\text{WheatYield}) = \beta_0 + \beta_1 \log(\text{Nitrogen}) + \beta_2 \log(\text{Area}) + \beta_3 \log(\text{Pesticides}) + \varepsilon$$

Where:
- $\beta_1, \beta_2, \beta_3$ are the elasticities of the three agricultural inputs
- No crop dummies are needed — I decided to analyse only one crop (Wheat)
- $\varepsilon$ is the error term, assumed normally distributed with mean zero

**Note:** YearTrend is intentionally excluded. The research question focuses on agricultural inputs, not on the technology trend over time.

In [ ]:
X_full, y = get_feature_target_split(fao)
y_log     = fao['LOG_Yield']

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42
)
_, _, y_log_train, y_log_test = train_test_split(
    X_full, y_log, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows | Features: {X_train.shape[1]}')

In [ ]:
ols_X_train = get_ols_features(X_train)
ols_X_test  = get_ols_features(X_test)

ols_model = fit_ols(ols_X_train, y_log_train)

ols_model.summary()

In [ ]:
ols_X_test_const = sm.add_constant(ols_X_test, has_constant='add')
ols_preds_log    = ols_model.predict(ols_X_test_const)
ols_metrics      = evaluate_model(y_log_test, ols_preds_log.values, 'OLS', log_target=True)
print(ols_metrics)

---
## 5. Model Diagnostics

In [ ]:
fig = plot_actual_vs_predicted(y_log_test, ols_preds_log.values, model_name='OLS — Wheat EU27')
plt.show()

---
## 6. Interpretation

### 6.1 Model performance
The OLS model with an R² of 0.516, explains almost 52% of the variance in log wheat yield across EU27 countries. The model is overall significant with a low F-statistic p-value, this means that the three inputs jointly explain a share of yield variation. However large part of the variance remains unexplained this is due to omitted variables.
The Adjusted R² of 0.514 is virtually identical to R², confirming that the result is not driven by overfitting. The sample covers 700 country-year observations across 27 EU member states from 1990 to 2023.

### 6.2 Significant predictors
All three predictors are statistically significant at the 1% level (p < 0.001).

**Nitrogen use** (LOG_NitrogenUse, coef = 0.759) is the strongest predictor. Since both variables are log-transformed, the coefficient is an elasticity: a 1% increase in nitrogen use is associated with a 0.76% increase in wheat yield, holding area and pesticides constant. This is consistent with the agronomic literature, nitrogen is the primary limiting nutrient for cereal crops in European conditions (Tilman et al., 2002).

**Area harvested** (LOG_AreaHarvested, coef = 0.104) is positive and significant, but the effect remains modest. A 1% increase in harvested area is associated with a 0.1% increase in yield, suggesting that scale plays a secondary role compared to input intensity.

**Pesticide use** (LOG_PesticideUse, coef = −0.106) has a negative and significant coefficient, this is a counterintuitive result. A 1% increase in harvested area is associated with a 0.1% decrease in yield. This result possible reflects reverse causality: countries with lower yields may apply more pesticides in response to higher pest pressure, creating a negative correlation in the data. This is a textbook example of the endogeneity problem discussed in Section 7, the negative sign should not be interpreted as pesticides reducing yield.

### 6.3 Residual diagnostics
The residuals deviate significantly from normality, as indicated by the Jarque-Bera and Omnibus tests (p-values ≈ 0), and exhibit negative skewness and high kurtosis, suggesting the presence of outliers or heavy tails.


## 7. Limitations & Future Work

### 7.1 Limitations
This analysis presents several limitations:

**1.Omitted variable bias** 
The model does not include important drivers of yield such as temperature, precipitation, and solar radiation. Since climate is correlated with both yield and input use (farmers in warmer, drier regions apply inputs differently), its omission creates omitted variable bias.

**2.Panel structure and fixed effects**
The dataset is a panel (country × year), but I model it as cross-sectional, ignoring unobserved time-invariant country differences such as soil quality, irrigation infrastructure, and farming culture. 

**3.Endogeneity of inputs**
Nitrogen use is chosen by farmers in anticipation of expected yield, creating reverse causality that is a classic endogeneity problem. OLS will therefore overestimate the causal effect of nitrogen on yield. Addressing this would require an instrumental variable (IV) approach, using variables that affect nitrogen use but not yield directly, such as fertilizer prices or distance to suppliers.

**4.Non-linearity**
OLS assumes a linear log-log relationship between inputs and yield, while the true nitrogen–yield relationship likely shows stronger diminishing returns at the high input levels typical of Western European agriculture. 

**5.Single crop limitation**
Focusing on wheat alone increases internal validity but limits generalisability. A multi-crop analysis with crop fixed effects would allow comparison across cereals.

### 7.2 Further extensions
Given these limitations, the most important extensions would be implementing a country fixed effects using a within-estimator to control for all stable country-level characteristics to exploit the panel structure of the data. Adding climate variables (temperature and precipitation by country-year) would reduce omitted variable bias and controlling for weather shocks. Finally, using fertilizer price indices as instrumental variables for nitrogen use would address the endogeneity problem and bring the nitrogen elasticity estimate closer to a causal interpretation.


## 8. Conclusion

This project uses three FAOSTAT datasets to model wheat yield across the 27 EU member states from 1990 to 2023. 
For this analysis, Wheat was selected as the focus crop because it is the most produced cereal in EU27 and one of only two cereals cultivated in all 27 member states, making it the natural choice for a cross-country regression analysis.
Using OLS regression on log-transformed yield, I find that nitrogen fertilizer use is the strongest predictor of wheat productivity (coef = 0.759, p < 0.001): a 1% increase in nitrogen use is associated with a 0.76% increase in yield. Area harvested has a modest positive effect (coef = 0.104), suggesting that scale plays a secondary role compared to input intensity. Pesticide use shows a negative coefficient (coef = −0.106), which I interpret as a sign of reverse causality rather than a causal effect; countries facing higher pest pressure apply more pesticides while also achieving lower yields.
The model explains almost 52% of wheat yield variance. However, the model does not control for country characteristics such as soil quality, farming infrastructure and does not apply a country fixed-effects. It does not include important drivers such as climate, soil quality, and country-level institutional differences.


## 9. Citations

- **Primary data:** FAO (2023). FAOSTAT Statistical Database. Rome: FAO. https://www.fao.org/faostat/
- **QCL:** FAO Crops & Livestock Products. https://www.fao.org/faostat/en/#data/QCL
- **EF:** FAO Fertilizers by Nutrient. https://www.fao.org/faostat/en/#data/RFB
- **EP:** FAO Pesticides Use. https://www.fao.org/faostat/en/#data/RP
- FAO, IFAD, UNICEF, WFP and WHO (2023). The State of Food Security and 
  Nutrition in the World 2023. Rome: FAO. https://doi.org/10.4060/cc3017en
- World Bank (2022). Agriculture, forestry, and fishing, value added (% of GDP). 
  World Development Indicators. https://data.worldbank.org/indicator/NV.AGR.TOTL.ZS
- Tilman et al. (2002). Agricultural sustainability and intensive production 
  practices. Nature, 418, 671–677.


## 10. Test section

In [ ]:
# This section tests functions used in the project
# 

from Tests.test_data_loader import test_get_feature_target_split
from src.data_loader import get_feature_target_split

TARGET = "Yield_t_ha"

test_get_feature_target_split(get_feature_target_split,TARGET)